In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

API key loaded successfully.


In [3]:
from langnchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

ModuleNotFoundError: No module named 'langnchain_community'

In [ ]:
DATA_PATH = "data"

loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load() 

print(f"Loaded {len(documents)} document page(s) from '{DATA_PATH}'")
if documents:
    print("\n--- Preview of first page ---")
    print(documents[0].page_content[:500])
    print("\n--- Metadata ---")
    print(documents[0].metadata)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document page(s) into {len(chunks)} chunk(s)")
if chunks:
    print("\n--- Preview of chunk 0 ---")
    print(chunks[0].page_content)
    print(f"\nChunk length: {len(chunks[0].page_content)} characters")

In [ ]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

sample_text = chunks[0].page_content if chunks else "LangChain makes it easy to build LLM applications."

vector = embeddings_model.embed_query(sample_text)

print(f"Embedding vector length: {len(vector)} dimensions")
print(f"First 10 values: {vector[:10]}")

In [ ]:
import numpy as np

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sentence_a = "The cat sat on the warm windowsill in the sun."
sentence_b = "A kitten was relaxing on a sunny window ledge."
sentence_c = "Quarterly revenue grew by twelve percent this year."

vec_a = embeddings_model.embed_query(sentence_a)
vec_b = embeddings_model.embed_query(sentence_b)
vec_c = embeddings_model.embed_query(sentence_c)

print("Similarity (A, B) — meaning-related sentences:", round(cosine_similarity(vec_a, vec_b), 4))
print("Similarity (A, C) — unrelated sentences:      ", round(cosine_similarity(vec_a, vec_c), 4))

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
DATA_PATH = "data"
PERSIST_DIR = "chroma_db"

loader = DirectoryLoader(DATA_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} page(s), split into {len(chunks)} chunk(s)")

In [ ]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=PERSIST_DIR
)

print(f"Vector store created and persisted to '{PERSIST_DIR}/'")
print(f"Total chunks stored: {vector_store._collection.count()}")


# vector_store = Chroma(
#     persist_directory=PERSIST_DIR,
#     embedding_function=embeddings_model
# )
# print(f"Reloaded existing store with {vector_store._collection.count()} chunks")

In [ ]:
def retrieve(query: str, k: int = 3):
    results = vector_store.similarity_search(query, k=k)
    print(f'Query: "{query}"')
    print(f'Top {k} matching chunk(s):\n')
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        print(f"--- Result {i} (source: {source}, page: {page}) ---")
        print(doc.page_content[:400])
        print()
    return results

# Try it with a question relevant to whatever you put in data/
test_results = retrieve("What is the main topic of this document?", k=3)